In [145]:
import pandas as pd

In [146]:
# SELECTED_PAGE = 215
SELECTED_PAGE = 5

In [147]:
path = r"C:\Users\samue\git\figma-rag\data\raw\figma_docs\manifest.jsonl"
df = pd.read_json(path, lines=True)

In [148]:
html_path = df.iloc[SELECTED_PAGE].raw_file_path
html_url = df.iloc[SELECTED_PAGE].source_url

In [149]:
html_path

'C:/Users/samue/git/figma-rag/data/raw/figma_docs/help_center/help-center-11786473291159-purchase-community-resources.html'

In [150]:
html_url

'https://help.figma.com/hc/en-us/articles/11786473291159-Purchase-Community-resources'

In [151]:
from bs4 import BeautifulSoup
from pathlib import Path

html = Path(html_path).read_text(encoding="utf-8")

soup = BeautifulSoup(html, "html.parser")

## Phase 1: Parse the HTML and remove global junk

In [152]:
removed = []

for selector in [
    "script",
    "style",
    "noscript",
    "template",
    "svg",
    "canvas",
    "iframe",
]:
    for node in soup.select(selector):
        removed.append({
            "selector": selector,
            "tag": node.name,
            "attrs": dict(node.attrs),
            "html": str(node),
            "text": node.get_text(separator="\n", strip=True),
        })
        node.decompose()


## Phase 2: Extract the main article node

In [153]:
import re

MAIN_CONTENT_SELECTORS = [
    ".article-body",
    "[data-swiftype-name='body']",
    ".article-content",
    ".article-info",
    "main article .article-body",
    "main article",
    "main",
]


def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text.strip()


def extract_main_node(soup: BeautifulSoup):
    for selector in MAIN_CONTENT_SELECTORS:
        node = soup.select_one(selector)

        if node:
            text = normalize_whitespace(node.get_text(" ", strip=True))

            if text:
                return node

    raise ValueError("Could not find a main content node")

def extract_header_node(soup: BeautifulSoup):
    node = soup.select_one(".article-header")
    if node:
        text = normalize_whitespace(node.get_text(" ", strip=True))

        if text:
            return node
    
    raise ValueError("Could not find a header node")

main_node = extract_main_node(soup)
header_node = extract_header_node(soup)

## Phase 3: Remove boilerplate inside the main node

In [154]:
REMOVE_INSIDE_MAIN_SELECTORS = [
    "nav",
    "form",
    "button",
    "select",
    "textarea",
    "input",

    ".breadcrumbs",
    ".sub-nav",
    ".article-footer",
    ".article-ticket-form",
    ".article-votes-question",
    "#feedback-success",

    "fl-course-navigator",
    "fl-metabar",
    "fl-mobile-metabar",

    ".hide",
    "[hidden]",
    "[aria-hidden='true']",
]


for selector in REMOVE_INSIDE_MAIN_SELECTORS:
    for node in main_node.select(selector):
        node.decompose()

## Phase 4: Extract useful content blocks
This keeps headings, paragraphs, list items, code, tables, image alt text, and captions.

In [155]:
# TEXT_TAGS = {
#     "h1", "h2", "h3", "h4", "h5", "h6",
#     "p",
#     "li",
#     "blockquote",
#     "pre",
#     "code",
#     "table",
# }


# def is_low_value_text(text: str) -> bool:
#     if not text:
#         return True

#     normalized = text.lower().strip()

#     low_value_exact = {
#         "yes",
#         "no",
#         "submit",
#         "search",
#         "sign up",
#         "contact sales",
#         "was this article helpful?",
#         "tell us more",
#         "submit article feedback",
#         "-- please choose an option --",
#     }

#     if normalized in low_value_exact:
#         return True

#     if len(text) < 3:
#         return True

#     return False


# content_blocks = []

# for node in main_node.find_all(list(TEXT_TAGS) + ["img"], recursive=True):
#     if node.name == "img":
#         alt = normalize_whitespace(node.get("alt", ""))

#         if alt and not is_low_value_text(alt):
#             content_blocks.append({
#                 "type": "image",
#                 "text": f"Image: {alt}",
#             })

#         continue

#     text = normalize_whitespace(node.get_text(" ", strip=True))

#     if is_low_value_text(text):
#         continue

#     content_blocks.append({
#         "type": node.name,
#         "text": text,
#     })

In [156]:
from urllib.parse import urljoin


TEXT_TAGS = {
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p",
    "li",
    "blockquote",
    "pre",
    "code",
    "table",
}


def is_low_value_text(text: str) -> bool:
    if not text:
        return True

    normalized = text.lower().strip()

    low_value_exact = {
        "yes",
        "no",
        "submit",
        "search",
        "sign up",
        "contact sales",
        "was this article helpful?",
        "tell us more",
        "submit article feedback",
        "-- please choose an option --",
    }

    if normalized in low_value_exact:
        return True

    if len(text) < 3:
        return True

    return False


def extract_links(node, base_url: str | None = None) -> list[dict]:
    links = []

    for a in node.find_all("a", href=True):
        link_text = normalize_whitespace(a.get_text(" ", strip=True))
        href = a["href"].strip()

        if not link_text or not href:
            continue

        # Ignore same-page anchors for now.
        if href.startswith("#"):
            continue

        # Convert relative URLs to absolute URLs if base_url is provided.
        url = urljoin(base_url, href) if base_url else href

        links.append({
            "text": link_text,
            "url": url,
        })

    return links


content_blocks = []

# Optional. Use the real page URL here if you have it.
source_url = "https://example.com/current-page"

for node in main_node.find_all(list(TEXT_TAGS) + ["img"], recursive=True):
    if node.name == "img":
        alt = normalize_whitespace(node.get("alt", ""))

        if alt and not is_low_value_text(alt):
            content_blocks.append({
                "type": "image",
                "text": f"Image: {alt}",
                "links": [],
            })

        continue

    text = normalize_whitespace(node.get_text(" ", strip=True))

    if is_low_value_text(text):
        continue

    links = extract_links(node, base_url=source_url)

    content_blocks.append({
        "type": node.name,
        "text": text,
        "links": links,
    })

In [157]:
content_blocks

[{'type': 'p',
  'text': 'You can purchase resources from creators on the Figma Community.',
  'links': []},
 {'type': 'p',
  'text': 'Resources on the Community may be sold using Figma’s payment platform or through a third-party platform. Figma is unable to provide purchase or refund support for resources sold on third-party platforms. You’ll need to contact the creator directly if you need assistance.',
  'links': []},
 {'type': 'p',
  'text': 'Note: We are unable to accept payments from users with Russian or Indian-based credit cards at this time.',
  'links': []},
 {'type': 'h2', 'text': 'Purchase paid files and templates', 'links': []},
 {'type': 'p',
  'text': 'You can preview a paid file before purchasing it. To preview a file, navigate to the file’s Community page and click Get free preview.',
  'links': []},
 {'type': 'p',
  'text': 'The checkout process differs depending on the payment platform used. You can identify if the file is sold using Figma’s payment platform or a thi

In [158]:
# Prepend header node to the content blocks

header_content_blocks = []

for node in header_node.find_all(list(TEXT_TAGS) + ["img"], recursive=True):
    if node.name == "img":
        alt = normalize_whitespace(node.get("alt", ""))

        if alt and not is_low_value_text(alt):
            content_blocks.append({
                "type": "image",
                "text": f"Image: {alt}",
            })

        continue

    text = normalize_whitespace(node.get_text(" ", strip=True))

    if is_low_value_text(text):
        continue

    header_content_blocks.append({
        "type": node.name,
        "text": text,
        "links": []
    })

header_content_blocks

[{'type': 'h1', 'text': 'Purchase Community resources', 'links': []}]

In [159]:
content_blocks = header_content_blocks + content_blocks

In [160]:
content_blocks[:4]

[{'type': 'h1', 'text': 'Purchase Community resources', 'links': []},
 {'type': 'p',
  'text': 'You can purchase resources from creators on the Figma Community.',
  'links': []},
 {'type': 'p',
  'text': 'Resources on the Community may be sold using Figma’s payment platform or through a third-party platform. Figma is unable to provide purchase or refund support for resources sold on third-party platforms. You’ll need to contact the creator directly if you need assistance.',
  'links': []},
 {'type': 'p',
  'text': 'Note: We are unable to accept payments from users with Russian or Indian-based credit cards at this time.',
  'links': []}]

## Phase 5: Preserve heading context

In [161]:
def add_heading_parents(content_blocks):
    heading_path = []
    content_blocks_w_parents = []

    for block in content_blocks:
        block_type = block.get("type")
        text = normalize_whitespace(block.get("text", ""))

        if is_low_value_text(text):
            continue

        if block_type in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            level = int(block_type[1])

            heading_path = [
                h for h in heading_path
                if h["level"] < level
            ]

            current_heading = {
                "level": level,
                "text": text,
            }

            heading_path.append(current_heading)

            content_blocks_w_parents.append({
                "type": "heading",
                "level": level,
                "text": text,
                "links": block.get("links", []),
                "headings": [h["text"] for h in heading_path],
                "heading_path": heading_path.copy(),
            })

        else:
            content_blocks_w_parents.append({
                "type": block_type,
                "text": text,
                "links": block.get("links", []),
                "headings": [h["text"] for h in heading_path],
                "heading_path": heading_path.copy(),
            })

    return content_blocks_w_parents

content_blocks_w_parents = add_heading_parents(content_blocks)

In [162]:
content_blocks_w_parents[:5]

[{'type': 'heading',
  'level': 1,
  'text': 'Purchase Community resources',
  'links': [],
  'headings': ['Purchase Community resources'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'}]},
 {'type': 'p',
  'text': 'You can purchase resources from creators on the Figma Community.',
  'links': [],
  'headings': ['Purchase Community resources'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'}]},
 {'type': 'p',
  'text': 'Resources on the Community may be sold using Figma’s payment platform or through a third-party platform. Figma is unable to provide purchase or refund support for resources sold on third-party platforms. You’ll need to contact the creator directly if you need assistance.',
  'links': [],
  'headings': ['Purchase Community resources'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'}]},
 {'type': 'p',
  'text': 'Note: We are unable to accept payments from users with Russian or Indian-based credit card

## Phase 6: Convert blocks to clean Markdown

In [163]:
[c for c in content_blocks_w_parents if c["type"] == "li"]

[{'type': 'li',
  'text': 'From the file’s Community, click Buy.',
  'links': [],
  'headings': ['Purchase Community resources',
   'Purchase paid files and templates'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'},
   {'level': 2, 'text': 'Purchase paid files and templates'}]},
 {'type': 'li',
  'text': 'Enter your payment details.',
  'links': [],
  'headings': ['Purchase Community resources',
   'Purchase paid files and templates'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'},
   {'level': 2, 'text': 'Purchase paid files and templates'}]},
 {'type': 'li',
  'text': 'Click Complete purchase.',
  'links': [],
  'headings': ['Purchase Community resources',
   'Purchase paid files and templates'],
  'heading_path': [{'level': 1, 'text': 'Purchase Community resources'},
   {'level': 2, 'text': 'Purchase paid files and templates'}]},
 {'type': 'li',
  'text': 'From the file’s Community page, click Open to access the file.',
  'links':

In [164]:
def blocks_to_markdown(blocks: list[dict]) -> str:
    lines = []

    for block in blocks:
        block_type = block["type"]
        text = block["text"]

        if block_type == "heading":
            level = block.get("level", 2)
            lines.append(f"{'#' * level} {text}")

        elif block_type == "li":
            lines.append(f"- {text}")

        elif block_type == "image":
            lines.append(f"[{text}]")

        else:
            lines.append(text)

    return "\n\n".join(lines)


markdown = blocks_to_markdown(content_blocks_w_parents)
print(markdown)

# Purchase Community resources

You can purchase resources from creators on the Figma Community.

Resources on the Community may be sold using Figma’s payment platform or through a third-party platform. Figma is unable to provide purchase or refund support for resources sold on third-party platforms. You’ll need to contact the creator directly if you need assistance.

Note: We are unable to accept payments from users with Russian or Indian-based credit cards at this time.

## Purchase paid files and templates

You can preview a paid file before purchasing it. To preview a file, navigate to the file’s Community page and click Get free preview.

The checkout process differs depending on the payment platform used. You can identify if the file is sold using Figma’s payment platform or a third-party platform from the file’s Community page:

[Image: Comparison of Figma (A) and third-party (B) payment buttons; A shows price, B redirects offsite.]

A. This file is sold through Figma’s payment 

In [165]:
# TODO add way to embed links when present